# Lotus Care Clinic Data Warehouse - Coursework Part C

This notebook profiles, cleans and transforms the raw clinic data
for loading into the PostgreSQL data warehouse.

Task 3: ETL Notebook - Profile, Cleanse and Integrate

Task 3.1: Import Libraries

In [40]:
import pandas as pd

Import Files

 Python OS module – directory listing

Data Loading with Pandas (CSV to DataFrame)

In [41]:
appointments = pd.read_csv("data/appointments_raw.csv")
doctors = pd.read_csv("data/doctors.csv")
clinics = pd.read_csv("data/clinics.csv")
patients = pd.read_csv("data/patients.csv")
patient_updates = pd.read_csv("data/patient_updates.csv")

 Check Raw Data Row Counts

This cell checks the number of rows and columns in each raw CSV file.

Expected output:
- appointments_raw.csv: 1,524 rows
- doctors.csv: 21 rows
- clinics.csv: 5 rows
- patients.csv: 300 rows
- patient_updates.csv: 3 rows


In [42]:
print("Appointments:", appointments.shape)
print("Doctors:", doctors.shape)
print("Clinics:", clinics.shape)
print("Patients:", patients.shape)
print("Patient Updates:", patient_updates.shape)

Appointments: (1524, 8)
Doctors: (21, 3)
Clinics: (5, 4)
Patients: (300, 4)
Patient Updates: (3, 3)


Profile Data

In [43]:
appointments.info()
appointments.isnull().sum()
appointments.duplicated().sum()


<class 'pandas.DataFrame'>
RangeIndex: 1524 entries, 0 to 1523
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   appointment_id        1524 non-null   str    
 1   patient_id            1524 non-null   str    
 2   doctor_id             1524 non-null   str    
 3   clinic_id             1524 non-null   str    
 4   appointment_date      1524 non-null   str    
 5   wait_minutes          1524 non-null   int64  
 6   consultation_minutes  1524 non-null   int64  
 7   fee                   1509 non-null   float64
dtypes: float64(1), int64(2), str(5)
memory usage: 135.6 KB


np.int64(0)

In [44]:
doctors.info()
doctors.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   doctor_id    21 non-null     str  
 1   doctor_name  21 non-null     str  
 2   specialty    21 non-null     str  
dtypes: str(3)
memory usage: 1.3 KB


doctor_id      0
doctor_name    0
specialty      0
dtype: int64

In [45]:
clinics.info()
clinics.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   clinic_id    5 non-null      str  
 1   clinic_name  5 non-null      str  
 2   city         5 non-null      str  
 3   province     5 non-null      str  
dtypes: str(4)
memory usage: 471.0 bytes


clinic_id      0
clinic_name    0
city           0
province       0
dtype: int64

In [46]:
patients.info()
patients.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   patient_id     300 non-null    str  
 1   gender         300 non-null    str  
 2   date_of_birth  300 non-null    str  
 3   district       300 non-null    str  
dtypes: str(4)
memory usage: 16.1 KB


patient_id       0
gender           0
date_of_birth    0
district         0
dtype: int64

In [47]:
patient_updates.info()
patient_updates.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   patient_id      3 non-null      str  
 1   new_district    3 non-null      str  
 2   effective_date  3 non-null      str  
dtypes: str(3)
memory usage: 270.0 bytes


patient_id        0
new_district      0
effective_date    0
dtype: int64

In [48]:
appointments.duplicated().sum()

np.int64(0)

Task 3.2: Remove Duplicate Bookings

In [49]:

print("Before:", len(appointments))
appointments = appointments.drop_duplicates(
    subset=['patient_id','doctor_id','clinic_id','appointment_date'],keep="first"
)

print("After:", len(appointments))

Before: 1524
After: 1500


Task 3.3: Standardize Date Format

In [50]:
appointments["appointment_date"].unique()[:15]

<ArrowStringArray>
[ '2025-04-25',  '2025-03-10',  '2025-08-19',  '2025-06-09',  '2025-08-20',
  '2025-04-29', '18-Oct-2025',  '2025-10-11',  '2025-03-04',  '2025-06-24',
  '2025-08-04',  '2025-10-17',  '2025-10-31',  '2025-11-25',  '2025-07-04']
Length: 15, dtype: str

In [51]:
appointments['appointment_date'] = pd.to_datetime(appointments['appointment_date'], errors='coerce')
appointments['appointment_date'].isnull().sum()

np.int64(60)

In [52]:
appointments["appointment_date"] = pd.to_datetime(appointments["appointment_date"], format="mixed")

In [53]:
appointments["appointment_date"].isnull().sum()

np.int64(60)

In [54]:
appointments["appointment_date"].unique()[:15]

<DatetimeArray>
['2025-04-25 00:00:00', '2025-03-10 00:00:00', '2025-08-19 00:00:00',
 '2025-06-09 00:00:00', '2025-08-20 00:00:00', '2025-04-29 00:00:00',
                 'NaT', '2025-10-11 00:00:00', '2025-03-04 00:00:00',
 '2025-06-24 00:00:00', '2025-08-04 00:00:00', '2025-10-17 00:00:00',
 '2025-10-31 00:00:00', '2025-11-25 00:00:00', '2025-07-04 00:00:00']
Length: 15, dtype: datetime64[us]

Task 3.4: Standardize Doctor Specialties

In [55]:
doctors["specialty"].value_counts()

specialty
General Medicine     4
Paediatrics          3
Cardiology           2
Orthopaedics         2
Neurology            2
Dermatology          2
dermatology          1
CARDIOLOGY           1
 Orthopaedics        1
cardiology           1
Neurology            1
general medicine     1
Name: count, dtype: int64

In [56]:
doctors['specialty'] = doctors['specialty'].str.strip().str.title()
print("After:", doctors['specialty'].value_counts())


After: specialty
General Medicine    5
Cardiology          4
Dermatology         3
Paediatrics         3
Orthopaedics        3
Neurology           3
Name: count, dtype: int64


Task 3.5: Validate Measures

In [57]:
appointments["wait_minutes"].describe()

count    1500.000000
mean       33.443333
std        33.972973
min       -30.000000
25%        10.000000
50%        24.000000
75%        45.000000
max       180.000000
Name: wait_minutes, dtype: float64

In [58]:
print("Wait minutes invalid:", (appointments['wait_minutes'] < 0).sum())

Wait minutes invalid: 8


In [59]:
appointments.loc[appointments['wait_minutes'] < 0, 'wait_minutes'] = pd.NA

In [60]:
print("Wait minutes invalid:", (appointments['wait_minutes'] < 0).sum())

Wait minutes invalid: 0


In [61]:
appointments["consultation_minutes"].describe()

count    1500.000000
mean       21.676667
std         7.913398
min         8.000000
25%        15.000000
50%        22.000000
75%        28.000000
max        35.000000
Name: consultation_minutes, dtype: float64

In [62]:
print("Consultation minutes invalid:", (appointments['consultation_minutes'] < 0).sum())

Consultation minutes invalid: 0


In [24]:
appointments["fee"].describe()

count    1485.000000
mean     3715.824916
std       877.649287
min      2500.000000
25%      3000.000000
50%      3500.000000
75%      4500.000000
max      5000.000000
Name: fee, dtype: float64

In [63]:
print("Fee invalid:", (appointments['fee'] < 0).sum())

Fee invalid: 0


Invalid rows drop and replace

In [64]:
medianfee = appointments['fee'].median()
appointments["fee"].fillna(medianfee, inplace=True)

C:\Users\devpa\AppData\Local\Temp\ipykernel_22248\173692274.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  appointments["fee"].fillna(medianfee, inplace=True)


0       3500.0
1       2500.0
2       4500.0
3       3000.0
4       3500.0
         ...  
1519    2500.0
1520    2500.0
1521    3500.0
1522    5000.0
1523    3000.0
Name: fee, Length: 1500, dtype: float64

In [65]:
appointments["fee"].isnull().sum()

np.int64(15)

Task 3.6: Join Appointments with Doctors

In [66]:
merged = appointments.merge(doctors, on="doctor_id", how="left")
print("Before join:", len(appointments))
print("After join:", len(merged))
print("Missing doctors:", merged['doctor_name'].isnull().sum())


Before join: 1500
After join: 1571
Missing doctors: 0


In [67]:
doctors["doctor_id"].duplicated().sum()

np.int64(1)

In [68]:
dup_id = doctors[doctors["doctor_id"].duplicated(keep=False)].sort_values("doctor_id")
print(dup_id)

   doctor_id              doctor_name         specialty
6        D07  Dr. Tharindu Rajapaksha  General Medicine
20       D07  Dr. Tharindu Rajapaksha  General Medicine


In [70]:
doctors = doctors.drop_duplicates(subset=["doctor_id"], keep="first")

merged = appointments.merge(doctors, on="doctor_id", how="left")

print("After join:", len(merged))


After join: 1500


Task 4: Surrogate Keys and Load

Task 4.1: Add Surrogate Keys

In [71]:
patients['patient_key'] = range(1, len(patients)+1)
doctors['doctor_key'] = range(1, len(doctors)+1)
clinics['clinic_key'] = range(1, len(clinics)+1)



In [72]:
dim_date = pd.DataFrame({"full_date": pd.date_range(start="2025-01-01", end="2025-12-31")})
dim_date["date_key"] = range(1, len(dim_date) + 1)
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["day_name"] = dim_date["full_date"].dt.day_name()

print(dim_date.shape)
dim_date.head()

(365, 6)


,full_date,date_key,year,quarter,month,day_name
0,2025-01-01,1,2025,1,1,Wednesday
1,2025-01-02,2,2025,1,1,Thursday
2,2025-01-03,3,2025,1,1,Friday
3,2025-01-04,4,2025,1,1,Saturday
4,2025-01-05,5,2025,1,1,Sunday


In [73]:
dim_date["day"] = dim_date["full_date"].dt.day
dim_date.head()

,full_date,date_key,year,quarter,month,day_name,day
0,2025-01-01,1,2025,1,1,Wednesday,1
1,2025-01-02,2,2025,1,1,Thursday,2
2,2025-01-03,3,2025,1,1,Friday,3
3,2025-01-04,4,2025,1,1,Saturday,4
4,2025-01-05,5,2025,1,1,Sunday,5


In [74]:
fact = appointments.merge(patients[["patient_id", "patient_key"]], on="patient_id", how="left")
fact = fact.merge(doctors[["doctor_id", "doctor_key"]], on="doctor_id", how="left")
fact = fact.merge(clinics[["clinic_id", "clinic_key"]], on="clinic_id", how="left")
fact = fact.merge(dim_date[["full_date", "date_key"]], left_on="appointment_date", right_on="full_date", how="left")
print(fact.shape)
fact.head()


(1500, 13)


,appointment_id,patient_id,doctor_id,clinic_id,appointment_date,wait_minutes,consultation_minutes,fee,patient_key,doctor_key,clinic_key,full_date,date_key
0,A00465,P0201,D15,C05,2025-04-25,3.0,28,3500.0,201,15,5,2025-04-25,115.0
1,A00270,P0011,D19,C01,2025-03-10,2.0,16,2500.0,11,19,1,2025-03-10,69.0
2,A00976,P0097,D20,C05,2025-08-19,13.0,29,4500.0,97,20,5,2025-08-19,231.0
3,A00657,P0255,D04,C04,2025-06-09,2.0,10,3000.0,255,4,4,2025-06-09,160.0
4,A00983,P0070,D15,C05,2025-08-20,40.0,30,3500.0,70,15,5,2025-08-20,232.0


In [75]:
fact_final = fact[["patient_key", "doctor_key", "clinic_key", "date_key", 
                     "wait_minutes", "consultation_minutes", "fee"]]
fact_final.head()

,patient_key,doctor_key,clinic_key,date_key,wait_minutes,consultation_minutes,fee
0,201,15,5,115.0,3.0,28,3500.0
1,11,19,1,69.0,2.0,16,2500.0
2,97,20,5,231.0,13.0,29,4500.0
3,255,4,4,160.0,2.0,10,3000.0
4,70,15,5,232.0,40.0,30,3500.0


In [85]:
fact_final["wait_minutes"] = fact_final["wait_minutes"].astype("Int64")
fact_final.head()

,patient_key,doctor_key,clinic_key,date_key,wait_minutes,consultation_minutes,fee
0,201,15,5,115.0,3,28,3500.0
1,11,19,1,69.0,2,16,2500.0
2,97,20,5,231.0,13,29,4500.0
3,255,4,4,160.0,2,10,3000.0
4,70,15,5,232.0,40,30,3500.0


Task 4.2: Export Cleaned CSVs

In [86]:
patients.to_csv("data/dim_patient.csv", index=False)
doctors.to_csv("data/dim_doctor.csv", index=False)
clinics.to_csv("data/dim_clinic.csv", index=False)


dim_date.to_csv("data/dim_date.csv", index=False)

fact_final.to_csv("data/fact_appointment.csv", index=False)

print("dim_patient:", patients.shape)
print("dim_doctor:", doctors.shape)
print("dim_clinic:", clinics.shape)
print("dim_date:", dim_date.shape)
print("fact_appointment:", fact.shape)


dim_patient: (300, 5)
dim_doctor: (20, 4)
dim_clinic: (5, 5)
dim_date: (365, 7)
fact_appointment: (1500, 13)


Task 7: A Patient Changes District

In [80]:
patient_updates

,patient_id,new_district,effective_date
0,P0022,Matara,2026-01-15
1,P0103,Colombo,2026-01-15
2,P0233,Kalutara,2026-01-15
